# Actin ring measurement: one field, step by step

This notebook takes **one microscope field** through the analysis behind the paper figure and ends with
the figure itself: the **Sato-weighted skeleton density** of the actin network, measured in rings from
the nucleus to the cell edge.

**The measurement.** For every pixel of a cell, $r = d_\text{nucleus} / (d_\text{nucleus} + d_\text{membrane})$
is 0 at the nucleus and 1 at the membrane. Actin fibres are enhanced with the multiscale Sato tubeness
filter, thresholded and thinned to a one-pixel skeleton. In each radial bin

$$\text{Sato-weighted skeleton density} = \frac{\sum_{\text{skeleton pixels in the bin}} \text{Sato response}}{\text{bin area (pixels)}}$$

so a thick, bright bundle counts more than a faint filament, and bins of different size are comparable.
Zones: **inner** $r < 0.33$, **mid** $0.33 \le r < 0.66$, **outer (cortical)** $r \ge 0.66$.

**Where the code is.**
- `old_pipeline/`: the TopoActin V3 code as it was (Cellpose segmentation, filters, batch pipeline).
- `replication/v3_paper.py`: the paper settings and one-field segmentation with the V3 classes.
- `replication/notebook_method.py`: the per-cell measurement of the interactive V3 notebook
  (Steps 6b/6c of `old_pipeline/TopoActin_V3_Interactive_DOWNSAMPLING.ipynb`), which made the paper figure.
- `replication/ring_plots.py`: the figures.

In [ ]:
import os, sys, time
from pathlib import Path

REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO / "replication"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from v3_paper import load_config, apply_paper_settings, analyze_field
import notebook_method as nm
import ring_plots

# ---- choose the field -------------------------------------------------------------------------
# One multi-channel Z-stack per field, channel order actin (CellMask), nuclei (Hoechst), brightfield.
# Default: the example C9_F13 of Exp10 (2026-01-13, plate 4), the BasR field of the paper figure.
# Other examples in examples/: D10_F02 (BasR), B5_F11 (KpSen), D11_F09 (control).
IMAGE = Path(os.environ.get("ACTIN_IMAGE", REPO / "examples" / "C9_F13_Z_stack_merged.tiff"))
OUT = Path(os.environ.get("ACTIN_OUT", "walkthrough_out"))
# ------------------------------------------------------------------------------------------------

cfg = load_config()        # replication/config_paper.json: settings of the paper runs
apply_paper_settings(cfg)  # the run-time switches the paper runs had (see v3_paper.py)
OUT.mkdir(parents=True, exist_ok=True)
print("Field:", IMAGE.name, "| found:", IMAGE.exists())

## 1. Segment cells and nuclei, apply the cell filters

Cellpose (pretrained `cpsam` model, Cellpose 4, no retraining) segments the cells on the actin + nuclei
channels and the nuclei on the nuclear channel. The V3 filters then keep cells with one valid nucleus that
do not touch the image border and have a plausible shape (solidity ≥ 0.70, eccentricity ≤ 0.95,
area ≥ 500 px). Takes about a minute on a 4 GB GPU.

In [ ]:
t0 = time.time()
field = analyze_field(IMAGE, cfg, skeletons=False)
seg = field["seg"]
kept = [c["cell_id"] for c in field["cell_filter"]["valid_cells"]]
nucleus_of = {c["cell_id"]: c["nucleus_id"] for c in field["cell_filter"]["valid_cells"]}
actin_mip = seg["actin_image_unadjusted"]   # maximum projection of the actin channel
nuclei_mip = seg["nuclei_image"]
print(f"{time.time() - t0:.0f} s: {len(np.unique(seg['cell_masks'])) - 1} cells found, {len(kept)} kept")
fig = ring_plots.show_segmentation(actin_mip, nuclei_mip, seg["cell_masks"], seg["nucleus_masks"], kept)
fig.savefig(OUT / f"{IMAGE.stem}_segmentation.png", dpi=200, bbox_inches="tight")

## 2. Sato tubeness filter on the whole field

As in the notebook: the actin maximum projection is rescaled to 0–1 (`local_adjust_image`, brightness 0,
contrast 1, gamma 1), smoothed (Gaussian σ = 0.7 px) and filtered with the multiscale Sato filter
(σ = 0.7, 0.96, 1.22, 1.48, 1.74, 2.0 px).

In [ ]:
image = nm.local_adjust_image(actin_mip)
sato_full = nm.sato_field(image)
fig, ax = plt.subplots(1, 2, figsize=(14, 7))
ax[0].imshow(image, cmap="gray"); ax[0].set_title("Actin, rescaled 0-1")
im = ax[1].imshow(sato_full, cmap="hot"); ax[1].set_title("Sato tubeness")
fig.colorbar(im, ax=ax[1], fraction=0.046)
for a in ax: a.axis("off")

## 3. Measure every kept cell

Per cell: threshold the Sato response at the 55th percentile of the cell's positive values, remove objects
under 3 px, close with a 1 px disk, skeletonize; compute $r$ from the cell's own nucleus; sum the Sato
response on the skeleton per bin and divide by the bin area.

In [ ]:
results = {}
for cid in tqdm(kept, desc="cells"):
    results[cid] = nm.analyze_cell(image, seg["cell_masks"], seg["nucleus_masks"], cid, nucleus_of[cid],
                                   sato_full=sato_full)
table = pd.DataFrame([{"cell_id": cid, "area_px": r["mask"].sum(), **r["row"]} for cid, r in results.items()])
table.insert(0, "image", IMAGE.stem)
table[["cell_id", "area_px", "inner_weighted_density", "mid_weighted_density", "cortical_weighted_density"]].round(4)

## 4. Choose a cell

Set `CELL_ID` to a number from the segmentation image. For the example C9_F13, cell 21 is the cell of
the paper figure slide (inner / mid / outer 0.0015 / 0.0028 / 0.0048). Set `None` to list the three
cells closest to the field's median cortical density and use the first.

In [ ]:
CELL_ID = 21 if IMAGE.name.startswith("C9_F13") else None
med = table.cortical_weighted_density.median()
candidates = table.iloc[(table.cortical_weighted_density - med).abs().argsort()[:3]]
print(f"field median cortical density: {med:.4f}; closest cells:")
print(candidates[["cell_id", "cortical_weighted_density", "area_px"]].round(4).to_string(index=False))
if CELL_ID is None or CELL_ID not in results:
    CELL_ID = int(candidates.cell_id.iloc[0])
cell = results[CELL_ID]
fig = ring_plots.eight_panels(cell, image)
fig.savefig(OUT / f"{IMAGE.stem}_cell{CELL_ID}_steps.png", dpi=200, bbox_inches="tight")

## 5. The paper figure

Original image, radial overlay of the chosen cell (zones, skeleton in white, outline in cyan), and the
distribution of the Sato-weighted skeleton density over 10 bins of $r$.

In [ ]:
fig = ring_plots.paper_figure(cell, actin_mip, nuclei_mip, seg["cell_masks"])
fig.savefig(OUT / f"{IMAGE.stem}_cell{CELL_ID}_figure.png", dpi=300, bbox_inches="tight")
fig.savefig(OUT / f"{IMAGE.stem}_cell{CELL_ID}_figure.svg", bbox_inches="tight")

## 6. Ten rings: nucleus + nine cytoplasm rings, merged 3-3-3

The same measurement in 10 rings: ring 0 is the nucleus, rings 1–9 split the cytoplasm into nine equal
steps of $r$. Merging rings 1–3, 4–6 and 7–9 (signal and area summed, not densities averaged) gives the
three zones without the nucleus. The zones above include the nucleus pixels ($r = 0$) in the inner zone,
so the outer zone is the same and the inner zone differs.

In [ ]:
fig = ring_plots.ten_rings(cell)
for z, ring in [("inner", "ring_inner3"), ("mid", "ring_mid3"), ("cortical", "ring_outer3")]:
    print(f"{z:9s} zone {cell['row'][z + '_weighted_density']:.4f}   rings merged {cell['row'][ring + '_weighted_density']:.4f}")

In [ ]:
table.to_csv(OUT / f"{IMAGE.stem}_cells.csv", index=False)
print("Saved figures and the cell table to", OUT.resolve())